# BD Plate Reader: training on Colab

Before you start:
1. Runtime menu, Change runtime type, choose a GPU (T4 is fine).
2. Your datasets must be in Google Drive at `MyDrive/bd-plate-reader/data/`, with their original names (the scripts also find them by name pattern, so small renames are fine):
   - `Bangla LPDB - A.zip`
   - `NightLPD-BD Annotated Bangladeshi Nighttime Vehicl.zip`
   - `bangladeshi-license-plate-recognition_-character-recognition-DatasetNinja.tar`
   - `Bangla Licence plate detection and recognition dataset.zip` (optional, no labels)
3. Run the cells from top to bottom. If Colab disconnects, run them again from the top. Training resumes from the last saved epoch because weights are saved to Drive after every epoch.

Everything is saved to `MyDrive/bd-plate-reader/outputs/`.

In [ ]:
# Settings. Change only if you know why.
REPO_URL = "https://github.com/Kanto065/bd-plate-reader.git"
BRANCH = "milestone-2"        # use "main" once the training branch is merged
EPOCHS_PLATE = 100            # stage 1, plate detector
EPOCHS_CHARS = 150            # stage 2, character detector
BATCH_PLATE = 16
BATCH_CHARS = 32

In [ ]:
# Connect Drive and check the GPU.
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi -L
!ls -lh "/content/drive/MyDrive/bd-plate-reader/data"

In [ ]:
# Get the code and install requirements.
import os
if not os.path.isdir("/content/bd-plate-reader"):
    !git clone -b {BRANCH} {REPO_URL} /content/bd-plate-reader
else:
    !git -C /content/bd-plate-reader pull
%cd /content/bd-plate-reader/training
!pip install -q -r requirements.txt

## Prepare the data
Unpacks the archives and builds the two training sets. The DatasetNinja photos are split by source photo: about 70 percent go to training, about 30 percent are held out as a separate test set, so no augmented copy of a test photo is ever used for training. Read the printed group counts and split sizes.

In [ ]:
# Unpack the datasets and build the two training sets (a few minutes).
!python organize_data.py
!python merge_datasets.py

In [ ]:
# Quick self check of the plate text logic.
!python -m pytest -q

## Stage 1: plate detector
Expect roughly 30 to 90 minutes on a T4 for 100 epochs.

In [ ]:
os.environ["BD_EPOCHS"] = str(EPOCHS_PLATE)
os.environ["BD_BATCH"] = str(BATCH_PLATE)
!python train.py plate

## Stage 2: character detector

In [ ]:
os.environ["BD_EPOCHS"] = str(EPOCHS_CHARS)
os.environ["BD_BATCH"] = str(BATCH_CHARS)
!python train.py chars

## Export to int8 TFLite
The first export installs TensorFlow packages and can take a few minutes.

In [ ]:
!python export.py

## Evaluate on the test split
Prints mAP for both models and the plate level exact match on test crops. The character model is scored on two held out sets, reported separately: the LPDB-A test split and the DatasetNinja held out photos. The Ninja number is the honest generalization check.
To also test the full pipeline on your own photos, put vehicle photos (and an optional truth.csv with columns file,text) in a Drive folder and set BD_PIPELINE_DIR before running this cell.

In [ ]:
# os.environ["BD_PIPELINE_DIR"] = "/content/drive/MyDrive/bd-plate-reader/my_photos"
!python evaluate.py

In [ ]:
# What was saved to Drive.
!ls -lh /content/drive/MyDrive/bd-plate-reader/outputs/models
!cat /content/drive/MyDrive/bd-plate-reader/outputs/eval_results.json

Download the two files in `MyDrive/bd-plate-reader/outputs/models/` (`plate_detector_int8.tflite` and `char_detector_int8.tflite`) and give them to Claude for Milestone 3.